# Omar Chatbot — Fine-tune LLaMA 3.1 8B Instruct with Unsloth

**Before running:** Runtime → Change runtime type → **A100 GPU** + High RAM

Steps:
1. Install dependencies
2. Upload `dataset.jsonl`
3. Load model + apply QLoRA
4. Train
5. Push to HuggingFace Hub (adapter + GGUF)

## 1. Install

In [ ]:
%%capture
!pip install unsloth
# Unsloth pulls in transformers, trl, peft, bitsandbytes automatically

## 2. Upload dataset

Run the cell below, then click **Choose Files** and upload `dataset.jsonl` from `~/meta-finetune/`.

In [ ]:
from google.colab import files
uploaded = files.upload()   # upload dataset.jsonl

import os
DATASET_PATH = list(uploaded.keys())[0]
print(f"Uploaded: {DATASET_PATH}")

# Quick sanity check
import json
with open(DATASET_PATH) as f:
    sample = json.loads(f.readline())
print(f"System prompt: {sample['messages'][0]['content']}")
print(f"First user turn: {sample['messages'][1]['content'][:100]}")

## 3. Load model with QLoRA

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LEN = 2048
DTYPE = None         # auto-detect (bfloat16 on A100)
LOAD_IN_4BIT = True  # QLoRA

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Meta-Llama-3.1-8B-Instruct",
    max_seq_length=MAX_SEQ_LEN,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)

print("Model loaded.")

In [ ]:
# Apply LoRA adapter
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                        # LoRA rank — 16 is a good default; try 32 if quality is lacking
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,               # keep equal to r for stable training
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",  # saves VRAM
    random_state=42,
    use_rslora=False,
)

model.print_trainable_parameters()

## 4. Prepare dataset

In [ ]:
from datasets import Dataset
import json

# Load JSONL
rows = []
with open(DATASET_PATH) as f:
    for line in f:
        line = line.strip()
        if line:
            rows.append(json.loads(line))

# Train / val split (95/5)
import random
random.seed(42)
random.shuffle(rows)
split = int(len(rows) * 0.95)
train_rows, val_rows = rows[:split], rows[split:]

train_dataset = Dataset.from_list(train_rows)
val_dataset   = Dataset.from_list(val_rows)

print(f"Train: {len(train_dataset):,}  |  Val: {len(val_dataset):,}")

In [ ]:
from unsloth.chat_templates import get_chat_template

# LLaMA 3 uses a different chat template than ChatML
tokenizer = get_chat_template(tokenizer, chat_template="llama-3")

def apply_template(examples):
    texts = [
        tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
        for msgs in examples["messages"]
    ]
    return {"text": texts}

train_dataset = train_dataset.map(apply_template, batched=True)
val_dataset   = val_dataset.map(apply_template, batched=True)

print(train_dataset[0]["text"][:500])

## 5. Train

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    dataset_num_proc=2,
    args=TrainingArguments(
        # --- core ---
        num_train_epochs=2,            # 2 epochs is a safe starting point
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8, # effective batch = 16
        # --- optimiser ---
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_steps=160,              # ~5% of steps
        optim="adamw_8bit",
        weight_decay=0.01,
        # --- precision ---
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        # --- logging / saving ---
        logging_steps=25,
        eval_strategy="no",            # disabled — Colab transformers version crashes eval loop
        save_strategy="steps",
        save_steps=200,
        save_total_limit=2,
        # --- misc ---
        seed=42,
        output_dir="/content/omar-qwen-checkpoints",
        report_to="none",
    ),
)

# Show memory usage before training
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 1)
max_memory = round(gpu_stats.total_memory / 1024**3, 1)
print(f"GPU: {gpu_stats.name}  |  VRAM: {max_memory} GB  |  Reserved: {start_gpu_memory} GB")

trainer_stats = trainer.train()

In [ ]:
## ⚡ Save to HuggingFace Hub IMMEDIATELY after training (before runtime can die)

HF_USERNAME = "obattisha"
REPO_NAME   = "omar-qwen2.5-7b"

from huggingface_hub import login
login()   # paste your write token when prompted — do NOT hardcode it here

model.save_pretrained_merged(
    f"/content/{REPO_NAME}",
    tokenizer,
    save_method="lora",
)
model.push_to_hub(f"{HF_USERNAME}/{REPO_NAME}", token=True)
tokenizer.push_to_hub(f"{HF_USERNAME}/{REPO_NAME}", token=True)
print(f"\nAdapter saved! https://huggingface.co/{HF_USERNAME}/{REPO_NAME}")

In [ ]:
## ⚡ Save to HuggingFace Hub IMMEDIATELY after training

HF_USERNAME = "obattisha"
REPO_NAME   = "omar-llama3.1-8b"

from huggingface_hub import login
login()   # paste your HF write token when prompted — do NOT hardcode it here

model.save_pretrained_merged(f"/content/{REPO_NAME}", tokenizer, save_method="lora")
model.push_to_hub(f"{HF_USERNAME}/{REPO_NAME}", token=True)
tokenizer.push_to_hub(f"{HF_USERNAME}/{REPO_NAME}", token=True)
print(f"\nAdapter saved! https://huggingface.co/{HF_USERNAME}/{REPO_NAME}")

In [ ]:
## ⚡ Save to HuggingFace Hub IMMEDIATELY after training (before runtime can die)
# Fill these in before training starts — don't wait until after!

HF_USERNAME = "your-hf-username"   # <-- your HF username
REPO_NAME   = "omar-qwen2.5-7b"    # <-- repo name (will be created if it doesn't exist)

from huggingface_hub import login
login()   # prompts for your HF write token (https://huggingface.co/settings/tokens)

model.save_pretrained_merged(
    f"/content/{REPO_NAME}",
    tokenizer,
    save_method="lora",   # adapter only — fast upload (~100MB)
)
model.push_to_hub(f"{HF_USERNAME}/{REPO_NAME}", token=True)
tokenizer.push_to_hub(f"{HF_USERNAME}/{REPO_NAME}", token=True)
print(f"\nSaved! https://huggingface.co/{HF_USERNAME}/{REPO_NAME}")

FastLanguageModel.for_inference(model)

SYSTEM = (
    "Your name is Omar Mohamed Battisha. You are Egyptian-American, based in the Bay Area, "
    "and a practicing Muslim. You studied Political Science and Economics at the University "
    "of Chicago, with a minor in Data Science. You were heavily involved in CHOMUN (UChicago's "
    "Model UN conference) as part of CHOSEC, the secretariat, and you competed in Arabic debate, "
    "including traveling to Qatar for tournaments.\n\n"
    "You're a Senior PM in AI and tech, with experience building 0-to-1 AI products — you led "
    "the launch of QuickBooks' first conversational AI agent at Intuit. You have a data science "
    "background and started your career in ML. You eat halal and are mostly pescatarian/vegetarian. "
    "You fence epee (aggressive style), summited Kilimanjaro, and have traveled to 35+ countries. "
    "You do woodworking. You avoid riba and engage seriously with Islamic philosophy and jurisprudence.\n\n"
    "You are progressive and politically engaged — you support Palestinian rights, believe in police "
    "reform, and care about working-class issues and climate action. You are not Zionist and you "
    "believe in gender equality. You speak Egyptian Arabic and naturally mix it into conversations.\n\n"
    "You have a dry, sardonic sense of humor. You're direct and warm with people you trust, but not "
    "performatively enthusiastic. You value effectiveness over hard work and ownership over "
    "effort-signaling. You're skeptical of received wisdom — you think common beliefs often contain "
    "mistruths.\n\n"
    "Respond naturally as yourself — in your own voice, with your own personality, vocabulary, and "
    "style. You're funny, direct, and warm. You use casual language (lol, lmao, fr, tbh, inshallah). "
    "If you don't know something about yourself, say so rather than guessing."
)

def chat(user_message, system=SYSTEM):
    messages = [
        {"role": "system",    "content": system},
        {"role": "user",      "content": user_message},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    )
    import torch
    if not isinstance(inputs, torch.Tensor):
        inputs = inputs["input_ids"]
    inputs = inputs.to("cuda")

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=256,
        temperature=0.8,
        top_p=0.9,
        repetition_penalty=1.1,
        do_sample=True,
    )
    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    return response

# Identity checks
for prompt in [
    "hey Omar, what's up?",
    "are you religious at all?",
    "what did you study in college?",
    "what do you do for work?",
    "what's your take on Palestine?",
    "are you a Zionist?",
    "you drink?",
]:
    print(f"User: {prompt}")
    print(f"Omar: {chat(prompt)}")
    print()

In [ ]:
FastLanguageModel.for_inference(model)

def chat(user_message, system=None):
    if system is None:
        system = (
            "Your name is Omar Battisha. You are chatting with someone. "
            "Respond naturally as yourself — in your own voice, with your own "
            "personality, vocabulary, and style."
        )
    messages = [
        {"role": "system",    "content": system},
        {"role": "user",      "content": user_message},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=256,
        temperature=0.8,
        top_p=0.9,
        repetition_penalty=1.1,
        do_sample=True,
    )
    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    return response

# Try a few prompts
for prompt in [
    "hey Omar, what's up?",
    "what do you do for fun?",
    "thoughts on the game last night?",
]:
    print(f"User: {prompt}")
    print(f"Omar: {chat(prompt)}")
    print()

## 7. Save

**Option A — save LoRA adapter only** (small, ~100MB, needs base model to run)  
**Option B — merge adapter into base model and save** (large, ~15GB, self-contained)  
**Option C — push to HuggingFace Hub** (recommended for hosting)

Run whichever fits your plan.

In [ ]:
# Push GGUF-quantized version for local/server hosting via ollama
model.push_to_hub_gguf(
    "obattisha/omar-llama3.1-8b-GGUF",
    tokenizer=tokenizer,
    quantization_method="q4_k_m",
    token=True,
)
print("GGUF pushed to https://huggingface.co/obattisha/omar-llama3.1-8b-GGUF")

In [ ]:
# Option C: push merged model to HuggingFace Hub
# First: get a write token at https://huggingface.co/settings/tokens

HF_USERNAME = "your-hf-username"   # <-- fill in
REPO_NAME   = "omar-qwen2.5-7b"    # <-- fill in

from huggingface_hub import login
login()   # will prompt for token

# Push adapter only (fast)
model.push_to_hub(f"{HF_USERNAME}/{REPO_NAME}", token=True)
tokenizer.push_to_hub(f"{HF_USERNAME}/{REPO_NAME}", token=True)
print(f"Pushed to https://huggingface.co/{HF_USERNAME}/{REPO_NAME}")

In [ ]:
# Option C (continued): also push a GGUF-quantized version for easy hosting
# This creates a Q4_K_M quantized file (~4GB) you can run with llama.cpp / ollama

model.push_to_hub_gguf(
    f"{HF_USERNAME}/{REPO_NAME}-GGUF",
    tokenizer=tokenizer,
    quantization_method="q4_k_m",   # good balance of speed vs quality
    token=True,
)
print("GGUF model pushed.")